# 8.2. Networks Using Blocks (VGG)
D2L의 Networks Using Blocks (VGG)장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. AlexNet에서 VGG로

### VGG (Visual Geometry Group)

AlexNet은 깊은 CNN이 이미지 분류에서 좋은 성능을 낼 수 있다는 것을 보여줬다. 하지만 AlexNet에는 한 가지 문제가 있었다. 각 층이 서로 다른 구조로 설계되어 CNN을 더 깊게 만드려면 어떤 규칙으로 만들어야할지 명확한 답이 없었다.

VGG는 이 문제를 해결하기 위해서 `Block`이라는 개념을 사용했다.

    Conv -> ReLU -> Conv -> ReLU -> Pool

같은 구조를 하나의 블록으로 만들고 이 블록을 반복해 깊은 CNN을 구성한다.

VGG의 핵심 아이디어는 이렇다.

- 작은 `3×3 Conv`를 반복한다.
- 여러 Conv를 처리한 뒤 Pooling한다.
- 이런 구조를 하나의 Block으로 만든다.
- Block을 반복해서 깊은 네트워크를 만든다.

## 2. 왜 Conv마다 Pooling하지 않는가?

### Conv마다 Pooling하면 생기는 문제
초기의 CNN을 단순하게 구성하면 다음과 같은 형태를 생각할 수 있다.
```text
Conv
↓
ReLU
↓
Pooling
↓
Conv
↓
ReLU
↓
Pooling
↓
...
```
하지만 Pooling을 너무 자주 사용하면 `H × W`가 너무 빠르게 작아진다.

예를 들어 입력이 `224 × 224`이고
매번 MaxPool을 이용해서 크기를 절반으로 줄인다면,
```text
224
↓
112
↓
56
↓
28
↓
14
↓
7
↓
3
↓
1
```
처럼 공간 정보가 빠르게 사라진다. 그래서 VGG에서는 Conv를 여러 번 수행한 다음 Pooling을 한 번 수행한다.

## 3. 왜 3 x 3 Conv를 반복할까?

VGG에서는 큰 Conv kernel 하나보다 작은 3 x 3 Conv를 여러 번 사용하는 것을 선호한다.

예를 들어서

3×3 Conv -> 3×3 Conv

를 연속으로 적용하면 한 출력 뉴런은 원본 입력의 약 `5×5` 영역을 볼 수 있다.

3×3 Conv 두 개 ≈ 5×5 Conv 한 개

정도의 receptive field를 가진다.

작은 Conv를 여러 번 사용하면 장점이 있다.

1. 큰 kernel보다 parameter를 줄일 수 있다.
2. 중간에 ReLU가 추가된다.
3. 따라서 더 복잡한 비선형 특징을 학습할 수 있다.

VGG의 설계 철학은 이렇다고 볼 수 있다.

> 큰 kernel 하나보다 작은 kernel을 여러 번 쌓자.

## 4. VGG Block 구조

하나의 VGG Block은 다음과 같이 구성된다.
```text
입력
↓
3×3 Conv + ReLU
↓
3×3 Conv + ReLU
↓
...
↓
2×2 MaxPool
↓
출력
```
Conv의 설정은 이렇다.

- kernel_size = 3
- padding = 1
- stride = 1

따라서 Conv를 통과해도 `H × W`는 유지된다.

마지막 MaxPool은

- kernel_size = 2
- stride = 2

를 사용한다.

따라서 Block 하나가 끝날 때마다 절반이 된다.

H → H / 2, W → W / 2

## 5. VGG Block 구현

In [ ]:
def vgg_block(num_convs, out_channels):
    layers = []

    # 여러 개의 Conv + ReLU
    for _ in range(num_convs): # num_convs = Block 안에 Conv를 몇 개 넣을지
        layers.append(
            nn.LazyConv2d(
                out_channels, # Conv가 몇 개의 feature map을 만들지
                kernel_size=3,
                padding=1
            )
        )
        layers.append(nn.ReLU())

    # 마지막에 H, W를 절반으로 감소
    layers.append(
        nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
    )

    return nn.Sequential(*layers)

예를 들어서

```py
vgg_block(2, 64)
```

라면

Conv(64채널) -> ReLU -> Conv(64채널) -> ReLU -> MaxPool 구조이다.

## 6. VGG는 Block을 조립해 만든다.

VGG에서는 Block의 설정만 정의한다. 예를 들어서 VGG-11의 구조는 이렇다.

```py
arch = (
    (1, 64), # (Conv 개수, 출력채널 수)
    (1, 128),
    (2, 256),
    (2, 512),
    (2, 512)
)
```

이런식으로 Block의 설정만 바꾸면 다양한 VGG 모델을 만들 수 있다.

VGG를 하나의 고정된 네트워크라기보단 같은 설계 원칙을 공유하는 네트워크 family로 보는게 좋다.

## 7. VGG-11 전체 구조

VGG-11

```text
Input
↓
Block 1 : Conv 1개, 64 channels
↓
Block 2 : Conv 1개, 128 channels
↓
Block 3 : Conv 2개, 256 channels
↓
Block 4 : Conv 2개, 512 channels
↓
Block 5 : Conv 2개, 512 channels
```

Conv layer 수는 8개이고 뒤에 Fully Connected Layer 3개가 있다. 8 Conv + 3 FC = 11 Layer 그래서 이 모델을 VGG-11이라고 부른다.

## 8. VGG 모델 구현

In [3]:
class VGG(nn.Module):
    def __init__(self, arch, num_classes=10):
        super().__init__()

        conv_blocks = []

        # arch를 따라 VGG Block 생성
        for num_convs, out_channels in arch:
            conv_blocks.append(
                vgg_block(num_convs, out_channels)
            )

        self.net = nn.Sequential(
            *conv_blocks,

            nn.Flatten(),

            nn.LazyLinear(4096),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.LazyLinear(4096),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.LazyLinear(num_classes)
        )

    def forward(self, x):
        return self.net(x)

전체 구조는 크게 특징 추출, 분류로 나눌 수 있다.

### 특징 추출

```text
VGG Block
↓
VGG Block
↓
VGG Block
↓
VGG Block
↓
VGG Block
```

Conv를 이용해 이미지의 특징을 추출한다.

### 분류

```text
Flatten
↓
Linear 4096
↓
ReLU
↓
Dropout
↓
Linear 4096
↓
ReLU
↓
Dropout
↓
Linear → Class
```

추출된 특징을 이용해 최종 클래스를 예측한다. 이 구조는 기본적으로 `Feature Extractor + Classifier` 형태이다.

## 9. 이미지 크기가 어떻게 변하는가?

In [4]:
arch = (
    (1, 64),
    (1, 128),
    (2, 256),
    (2, 512),
    (2, 512)
)

model = VGG(arch)

X = torch.randn(1, 3, 224, 224)

for layer in model.net:
    X = layer(X)

    print(
        layer.__class__.__name__,
        X.shape
    )

Sequential torch.Size([1, 64, 112, 112])
Sequential torch.Size([1, 128, 56, 56])
Sequential torch.Size([1, 256, 28, 28])
Sequential torch.Size([1, 512, 14, 14])
Sequential torch.Size([1, 512, 7, 7])
Flatten torch.Size([1, 25088])
Linear torch.Size([1, 4096])
ReLU torch.Size([1, 4096])
Dropout torch.Size([1, 4096])
Linear torch.Size([1, 4096])
ReLU torch.Size([1, 4096])
Dropout torch.Size([1, 4096])
Linear torch.Size([1, 10])


깊은 층으로 갈수록 이미지의 정확한 위치 정보는 압축되고 더 많은 종류의 특징을 표현하는 feature map이 만들어진다.

## 10. 왜 H, W는 줄이고 Channel은 높이나?

처음에는 [3, 224, 224]처럼 공간 정보가 매우 많다. 하지만 깊은 층으로 갈수록 모델이 필요한 것은 원래 이미지 자체보다는 이미지에서 추출된 특징이다.

예를 들어 사람을 분류한다고 치면

초반 Conv는 선, 경계, 방향, 색 변화 같은 단순한 특징을 학습할 수 있다.

더 깊은 층에서는 이런 특징을 조합해 눈, 얼굴, 손, 옷 같은 복잡한 특징을 표현할 수 있다.

그래서 공간 크기 H x W는 점점 줄이고 다양한 특징을 저장할 수 있도록 채널 수를 증가시킨다.

## 11. Fashion-MNIST에서는 작은 VGG 사용

VGG-11은 상당히 크다. 그래서 Fashion-MNIST와 같은 간단한 데이터에 그대로 사용하는 건 계산량이 지나치게 크다.

```text
원본

64 -> 128 -> 256 -> 512 -> 512

축소

16 -> 32 -> 64 -> 128 -> 128
```

Block 기본 구조는 같고 채널 수만 줄였다.

In [5]:
small_arch = (
    (1, 16),
    (1, 32),
    (2, 64),
    (2, 128),
    (2, 128)
)

model = VGG(
    arch=small_arch,
    num_classes=10
)

model

VGG(
  (net): Sequential(
    (0): Sequential(
      (0): LazyConv2d(0, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): LazyConv2d(0, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): LazyConv2d(0, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): LazyConv2d(0, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (3): Sequential(
      (0): LazyConv2d(0, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): LazyConv2d(0, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
      (4): MaxPool2d(ker

## 12. 오늘의 정리

- VGG는 CNN을 **Block 단위로 설계**하기 시작한 대표적인 모델이다.
- 하나의 VGG Block은 여러 `3×3 Conv + ReLU`와 하나의 MaxPool로 구성된다.
- Conv에서는 `padding=1`을 사용해 H, W를 유지한다.
- Block 마지막의 `2×2 MaxPool, stride=2`가 H와 W를 절반으로 줄인다.
- 작은 `3×3 Conv`를 여러 번 쌓으면 큰 kernel과 비슷한 receptive field를 만들 수 있다.
- Conv를 여러 번 수행한 뒤 Pooling하기 때문에 깊은 네트워크를 만들 수 있다.
- VGG에서는 깊어질수록 `H, W는 감소`하고 `Channel은 증가`한다.
- VGG-11은 Conv 8개 + Fully Connected 3개 = 총 11개 layer이다.
- `arch = (Conv 개수, Channel 수)` 형태로 Block 구성을 정의할 수 있다.
- VGG의 가장 중요한 아이디어는 특정 숫자보다 **반복 가능한 Block으로 CNN을 설계한다는 것**이다.